# Entrainer le modele hybride sur Colab

Entraine CLIP + SmolLM2 en trois phases, avec les checkpoints sur Drive.

Piste abandonnee : tout a fini par tourner sur Kaggle. Le notebook est garde pour memoire.

Avant de lancer : runtime sur GPU, puis remplir la cellule de config (`REPO_URL`, et
eventuellement le zip de donnees).

## Config

Change these values, then run this cell.

In [ ]:
# A remplir
REPO_URL = "https://github.com/YOUR_USER/price-tracker.git"  # None pour sauter le clone
REPO_BRANCH = "ocr_worker_module"

DRIVE_ROOT = "/content/drive/MyDrive/receipt_vlm"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"

# Les phases a jouer. Sauter une phase suppose que son checkpoint est deja sur Drive.
RUN_PHASE_1 = True
RUN_PHASE_2 = True
RUN_PHASE_3 = True
RUN_EXPORT = True

# Le zip des photos et des labels, sur Drive. Necessaire pour les phases 2 et 3, qui ont
# besoin des vraies photos. La phase 1 s'en passe.
DATA_ZIP_ON_DRIVE = None

# A passer a True si le GPU sature
FORCE_SMALL_BATCH = False

## Monter Drive, et verifier le GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

for d in (DRIVE_ROOT, CHECKPOINT_DIR,
          f"{DRIVE_ROOT}/images_tickets_caisse", f"{DRIVE_ROOT}/real_labels"):
    Path(d).mkdir(parents=True, exist_ok=True)

import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

## Recuperer le code

In [ ]:
import subprocess
import sys

WORK = Path("/content/price-tracker")
TRAIN_PKG = WORK / "dev_ocr" / "vlm_training"

if REPO_URL:
    if not WORK.is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(WORK)])
    else:
        subprocess.check_call(["git", "-C", str(WORK), "pull"])
else:
    assert TRAIN_PKG.is_dir(), (
        "Set REPO_URL or upload the repo and adjust WORK path. "
        "Expected dev_ocr/vlm_training under WORK."
    )

os.chdir(TRAIN_PKG)
print("Working directory:", os.getcwd())

## Installer les dependances

In [ ]:
!pip install -q -r requirements-training.txt "tokenizers>=0.22,<=0.23"
!pip install -q -e .
!pip install -q -e ..

## Depaqueter les photos et leurs labels (phases 2 et 3)

In [ ]:
import zipfile

labels_ok = (Path(DRIVE_ROOT) / "real_labels" / "splits.json").is_file()
n_images = len(list((Path(DRIVE_ROOT) / "images_tickets_caisse").glob("*.*")))

if DATA_ZIP_ON_DRIVE and Path(DATA_ZIP_ON_DRIVE).is_file():
    print(f"Unzipping {DATA_ZIP_ON_DRIVE} → {DRIVE_ROOT}")
    with zipfile.ZipFile(DATA_ZIP_ON_DRIVE) as zf:
        zf.extractall(DRIVE_ROOT)
    labels_ok = (Path(DRIVE_ROOT) / "real_labels" / "splits.json").is_file()
    n_images = len(list((Path(DRIVE_ROOT) / "images_tickets_caisse").glob("*.*")))

print(f"real_labels/splits.json: {labels_ok}")
print(f"images_tickets_caisse: {n_images} files")
if (RUN_PHASE_2 or RUN_PHASE_3) and not labels_ok:
    print("WARNING: phases 2–3 need real data. Upload zip and set DATA_ZIP_ON_DRIVE.")

## Entrainer les trois phases

In [ ]:
import subprocess

def run_train(config: str, resume: str | None = None) -> None:
    cmd = [sys.executable, "scripts/train.py", "--config", config]
    if resume:
        cmd += ["--resume", resume]
    print(" ".join(cmd), flush=True)
    subprocess.check_call(cmd)

p1 = f"{CHECKPOINT_DIR}/phase1_best.pt"
p2 = f"{CHECKPOINT_DIR}/phase2_best.pt"
p3 = f"{CHECKPOINT_DIR}/phase3_best.pt"

if FORCE_SMALL_BATCH:
    import yaml
    for name in ("phase1_colab.yaml", "phase2_colab.yaml", "phase3_colab.yaml"):
        path = Path("configs") / name
        cfg = yaml.safe_load(path.read_text()) or {}
        cfg["batch_size"] = 4
        path.write_text(yaml.dump(cfg, default_flow_style=False))
    print("Patched colab configs: batch_size=4")

if RUN_PHASE_1:
    run_train("configs/phase1_colab.yaml")
else:
    assert Path(p1).is_file(), f"Missing {p1} — enable RUN_PHASE_1 or upload checkpoint"
    print(f"Skipping phase 1 ({p1} exists)")

if RUN_PHASE_2:
    run_train("configs/phase2_colab.yaml", resume=p1)
else:
    assert Path(p2).is_file(), f"Missing {p2}"
    print(f"Skipping phase 2 ({p2} exists)")

if RUN_PHASE_3:
    run_train("configs/phase3_colab.yaml", resume=p2)
else:
    assert Path(p3).is_file(), f"Missing {p3}"
    print(f"Skipping phase 3 ({p3} exists)")

## Exporter le checkpoint fusionne

In [ ]:
merged = f"{DRIVE_ROOT}/receipt_vlm_500m_merged.pt"

if RUN_EXPORT:
    subprocess.check_call([
        sys.executable, "scripts/export_checkpoint.py",
        "--checkpoint", p3,
        "--output", merged,
    ])
    print(f"Done. Download from Drive: {merged}")
else:
    print("Export skipped (RUN_EXPORT=False)")

## Verification rapide sur une photo

In [ ]:
from pathlib import Path
import os

photos = sorted(Path(f"{DRIVE_ROOT}/images_tickets_caisse").glob("*.jpg"))
if photos and Path(merged).is_file():
    os.environ["RECEIPT_OCR_BACKEND"] = "vlm"
    os.environ["RECEIPT_VLM_MODEL"] = "receipt-vlm-500m"
    os.environ["RECEIPT_VLM_MODE"] = "json"
    os.environ["RECEIPT_VLM_MODEL_PATH"] = merged

    import json
    from receipt_ocr import extract_receipt

    sample = photos[0]
    result = extract_receipt(str(sample))
    print(f"Sample: {sample.name}")
    print(json.dumps(result, indent=2, ensure_ascii=False)[:1200], "...")
else:
    print("Skip: need merged checkpoint and at least one photo in images_tickets_caisse")